# Aircraft XML to Excel Preprocessing

This notebook reads aircraft data from an XML file, extracts relevant fields, computes the tire contact area, and exports the results to an Excel file.

In [36]:
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
from pathlib import Path

xml_path = Path("input_data/aircraft.xml")
tree = ET.parse(xml_path)
root = tree.getroot()

NS_F = "http://schemas.datacontract.org/2004/07/FaarFieldModel"
NS_A = "http://schemas.microsoft.com/2003/10/Serialization/Arrays"
NS_XSI = "http://www.w3.org/2001/XMLSchema-instance"

airplanes = root.find(f".//{{{NS_F}}}Airplanes")
assert airplanes is not None, "Could not find <Airplanes> in the XML."

def get_scalar(parent, tag, default=np.nan, cast=float):
    """Get text of a simple element <tag>value</tag> under parent."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return default
    # handle xsi:nil="true"
    if el.attrib.get(f"{{{NS_XSI}}}nil", "").lower() == "true":
        return default
    if el.text is None:
        return default
    try:
        return cast(el.text.strip())
    except Exception:
        return default

def get_us(parent, tag, default=np.nan, cast=float):
    """Get the <us> child value under a unit-wrapped element <tag><si>..</si><us>..</us></tag>."""
    el = parent.find(f"{{{NS_F}}}{tag}")
    if el is None:
        return default
    us = el.find(f"{{{NS_F}}}us")
    if us is None or us.text is None:
        return default
    try:
        return cast(us.text.strip())
    except Exception:
        return default

In [ ]:
rows = []
for ap in list(airplanes):
    # only keep the AirplaneInfo blocks
    if ap.tag != f"{{{NS_A}}}anyType":
        continue
    if ap.attrib.get(f"{{{NS_XSI}}}type") != "AirplaneInfo":
        continue

    name = ap.find(f"{{{NS_F}}}Name")
    name = name.text.strip() if (name is not None and name.text) else ""

    # Requested fields
    gw_lbs = get_us(ap, "_GrossWeight")          # "Gross Taxi Weight (lbs)"
    tire_area = get_us(ap, "TireArea")
    tire_len = get_us(ap, "TireLength")
    tire_wid = get_us(ap, "TireWidth")

    # TirePressureF is often nil; default to 0 per your requirement
    tire_pressure = get_scalar(ap, "Cp", default=0.0)
    mg_percent = get_scalar(ap, "MgPercent")
    mg_percent_pcn = get_scalar(ap, "MgPercentPCN")

    num_gear = get_scalar(ap, "NumberGear", cast=int)
    num_tracks = get_scalar(ap, "NumberTireTracks", cast=int)
    num_wheels = get_scalar(ap, "NumberWheels", cast=int)

    rows.append({
        "Airplane Name": name,
        "Gross Taxi Weight (lbs)": gw_lbs,
        "Tire Pressure (psi)": tire_pressure,
        "Percent GW on Gear": mg_percent,
        "MgPercentPCN": mg_percent_pcn,
        "Number Gear": num_gear,
        "Number Tire Tracks": num_tracks,
        "Number Wheels": num_wheels,
        "Tire Contact Width (in.)": tire_wid,
        "Tire Contact Length (in.)": tire_len,
        "Tire Contact Area (in.^2)": tire_area,
    })

df = pd.DataFrame(rows)

# Calculate load per tire (lbs)
df["Load (lbs)"] = (df["Gross Taxi Weight (lbs)"] * df["MgPercentPCN"]) / df["Number Tire Tracks"]
df["Load (kN)"] = df["Load (lbs)"] * 0.00444822


# Optional: round the contact geometry to match your example output style
df["Tire Contact Width (in.)"] = df["Tire Contact Width (in.)"].round(1)
df["Tire Contact Length (in.)"] = df["Tire Contact Length (in.)"].round(1)
df["Tire Contact Area (in.^2)"] = df["Tire Contact Area (in.^2)"].round(1)

out_path = Path("input_data/aircraft.xlsx")
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Aircraft", index=False)

In [40]:
# Visualize the first rows of the dataframe
df.head()

,Airplane Name,Gross Taxi Weight (lbs),Tire Pressure (psi),Percent GW on Gear,MgPercentPCN,Number Gear,Number Tire Tracks,Number Wheels,Tire Contact Width (in.),Tire Contact Length (in.),Tire Contact Area (in.^2)
0,SWL-2,2000.0,0.0,1.0,1.0,1,1,1,7.3,11.7,67.0
1,SWL-5,5000.0,0.0,1.0,1.0,1,1,1,9.4,15.0,111.0
2,SWL-10,10000.0,0.0,1.0,1.0,1,1,1,12.6,20.2,200.0
3,Single Wheel 2,2000.0,0.0,1.0,0.5,1,1,1,0.0,0.0,0.0
4,Single Wheel 5,5000.0,0.0,1.0,0.5,1,1,1,0.0,0.0,0.0
